# Create train and test set

This notebook processes cholera outbreaks and essential climate variable data, adds additional features, merges all the data and finally saves a dataset that can be then used to create train and test sets for modeling.

In [ ]:
# import packages
import gc
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from geopandas.tools import geocode

In [ ]:
# make list of desired years
years = list(np.arange(2010, 2019))

In [ ]:
def process_ecv(evc_data, ecv):
    """
    This functions resamples given ecv data to the district level and calculates the areal mean of the ecv.
    It then add several lag variables.
    Finally, it drops rows with missing values and returns a dataframe.
    """

    ecv_data_processed = (
        evc_data.groupby(["district", "year", "month"]).aggregate("mean").reset_index()
    )

    ecv_data_processed[ecv + "_1m_l"] = ecv_data_processed.groupby(["district"])[ecv].shift(
        1
    )  # 1 month lag
    ecv_data_processed[ecv + "_c_1m_l"] = (
        (ecv_data_processed[ecv] / ecv_data_processed[ecv + "_1m_l"]) - 1
    ) * 100  # rate of change between current month and 1 month lag
    ecv_data_processed[ecv + "_d_1m_l"] = ecv_data_processed[ecv + "_c_1m_l"].apply(
        lambda x: 1 if x > 1 else 0
    )  # dummy indicating whether current month is higher or lower than 1 month lag
    ecv_data_processed[ecv + "_2m_l"] = ecv_data_processed.groupby(["district"])[ecv].shift(
        2
    )  # 2 month lag
    ecv_data_processed[ecv + "_c_2m_l"] = (
        (ecv_data_processed[ecv] / ecv_data_processed[ecv + "_2m_l"]) - 1
    ) * 100  # rate of change between current month and 2 month lag
    ecv_data_processed[ecv + "_d_2m_l"] = ecv_data_processed[ecv + "_c_2m_l"].apply(
        lambda x: 1 if x > 1 else 0
    )  # dummy indicating whether current month is higher or lower than 2 month lag

    # null values result naturally for the 1 month lag variable for the first month of 2010
    # null values result naturally for the 2 month lag variable for the first two months of 2010
    # these first two months of 2010 are consequently dropped for all districts
    ecv_data_processed = ecv_data_processed.dropna().reset_index(drop=True)

    return ecv_data_processed

## Cholera outbreaks

First, we load the cholera outbreaks data and explore it.

In [ ]:
path = "../data/cholera_outbreaks"

In [ ]:
file = "monthly_cholera_outbreaks_per_district_2010_2018.shp"

In [ ]:
%%time

outbreaks = gpd.read_file(os.path.join(path, file))

In [ ]:
outbreaks.crs

In [ ]:
outbreaks.shape

In [ ]:
outbreaks.info()

In [ ]:
outbreaks.head()

In [ ]:
outbreaks[["district", "geometry"]].drop_duplicates().plot()

### Buffer district borders by 1 degree

It might be a good idea to reproject the data to a CRS that is suited to this specific region (e.g. https://epsg.io/7755). But which one exactly?

In [ ]:
# reproject crs
outbreaks = outbreaks.to_crs(epsg=7755)

In [ ]:
outbreak_geometries = outbreaks[["district", "geometry"]].drop_duplicates().reset_index(drop=True)

In [ ]:
# buffer district borders by 1 degree/111000 meters
outbreak_geometries_buffered = outbreak_geometries.copy()
outbreak_geometries_buffered["geometry"] = outbreak_geometries_buffered["geometry"].buffer(111000)

In [ ]:
outbreak_geometries_buffered.head()

In [ ]:
outbreak_geometries_buffered.plot()

### Prepare polygon to extract relevant oceanic ECV data

#### Get polygon of India

In [ ]:
india = gpd.read_file("../data/cholera_outbreaks/gadm36_IND_shp/gadm36_IND_0.shp")

In [ ]:
india.crs

In [ ]:
india = india.to_crs(epsg=7755)

In [ ]:
india.crs

In [ ]:
india.head()

In [ ]:
india.plot()

#### Buffer polygon of India by 1 degree

In [ ]:
india_buffered = india.copy()
india_buffered["geometry"] = india_buffered["geometry"].buffer(111000)

In [ ]:
india_buffered.head()

In [ ]:
india_buffered.plot()

#### Get area that is relevant for oceanic ECV data

In [ ]:
india.equals(india_buffered)

In [ ]:
india_buffered.equals(india)

In [ ]:
india.difference(india_buffered)

In [ ]:
india_buffered.difference(india)

In [ ]:
india_buffered.difference(india)[0]

In [ ]:
india_buffered.difference(india).plot()

In [ ]:
india_buffered_difference = india_buffered.copy()
india_buffered_difference = india_buffered_difference.append(
    {"GID_0": "IND", "NAME_0": "India", "geometry": india_buffered.difference(india)[0]},
    ignore_index=True,
)
india_buffered_difference = india_buffered_difference.tail(1).reset_index(drop=True)
india_buffered_difference

In [ ]:
india_buffered_difference.info()

## Resample ECV data to district level, compute areal means and add lag features

### Sea surface salinity

First, we load a sample of the data and explore it.

In [ ]:
path = "../data/sea_surface_salinity"

In [ ]:
file = "monthly_sss_" + str(years[0]) + ".shp"

In [ ]:
%%time

sss = gpd.read_file(os.path.join(path, file))

In [ ]:
sss.shape

In [ ]:
sss.info()

In [ ]:
sss.head()

Now we process all the data.

In [ ]:
%%time

sss = pd.DataFrame(columns=["district", "year", "month", "sss"])

for year in years:
    print("Processing {}...".format(year))

    # read data into geodataframe
    sss_temp = gpd.read_file(os.path.join(path, "monthly_sss_" + str(year) + ".shp"))

    # reproject data to local crs
    sss_temp = sss_temp.to_crs(epsg=7755)

    # extract data over coastal area extending from district shoreline to one degree offshore
    sss_temp = sss_temp.sjoin(
        india_buffered_difference, how="inner", predicate="within"
    ).reset_index(drop=True)
    sss_temp = sss_temp.drop(["index_right", "GID_0", "NAME_0"], axis=1)

    # spatial join with buffered districts because sss is an oceanic variable
    sss_temp = sss_temp.sjoin(
        outbreak_geometries_buffered, how="inner", predicate="within"
    ).reset_index(drop=True)
    sss_temp = sss_temp.drop("index_right", axis=1)

    # append processed data to previously created dataframe
    sss = sss.append(sss_temp[["district", "year", "month", "sss"]]).reset_index(drop=True)

In [ ]:
%%time

# aggregate data and add lag variables
sss_district_level = process_ecv(sss, "sss")

In [ ]:
# add unbuffered district geometries
sss_district_level = pd.merge(
    sss_district_level, outbreak_geometries, how="inner", on="district"
).reset_index(drop=True)

In [ ]:
# transform dataframe to geodataframe
sss_district_level = gpd.GeoDataFrame(
    sss_district_level, crs="EPSG:7755", geometry=sss_district_level["geometry"]
)

Check out the processed file to make sure the output is as desired.

In [ ]:
sss_district_level.shape

In [ ]:
sss_district_level.info()

In [ ]:
sss_district_level.head()

In [ ]:
sss_district_level["district"].nunique()

In [ ]:
# only coastal districts that interact with the coverage of sss are included
sss_district_level[["geometry"]].drop_duplicates().to_crs(epsg=4326).plot()

In [ ]:
# sss is included for coastal districts only, here shown exemplary for one month
sss_temp[sss_temp["month"] == 1].to_crs(epsg=4326).plot()

In [ ]:
# free memory
del [sss_temp, sss]
gc.collect()

### Chlorophyll-a concentration

First, we load a sample of the data and explore it.

In [ ]:
path = "../data/chlorophyll_a_concentration"

In [ ]:
file = "monthly_chlora_" + str(years[0]) + ".shp"

In [ ]:
%%time

chlora = gpd.read_file(os.path.join(path, file))

In [ ]:
chlora.shape

In [ ]:
chlora.info()

In [ ]:
chlora.head()

Now we process all the data.

In [ ]:
%%time

chlora = pd.DataFrame(columns=["district", "year", "month", "chlora"])

for year in years:
    print("Processing {}...".format(year))

    # read data into geodataframe
    chlora_temp = gpd.read_file(os.path.join(path, "monthly_chlora_" + str(year) + ".shp"))

    # reproject data to local crs
    chlora_temp = chlora_temp.to_crs(epsg=7755)

    # extract data over coastal area extending from district shoreline to one degree offshore
    chlora_temp = chlora_temp.sjoin(
        india_buffered_difference, how="inner", predicate="within"
    ).reset_index(drop=True)
    chlora_temp = chlora_temp.drop(["index_right", "GID_0", "NAME_0"], axis=1)

    # spatial join with buffered districts because chlora is an oceanic variable
    chlora_temp = chlora_temp.sjoin(
        outbreak_geometries_buffered, how="inner", predicate="within"
    ).reset_index(drop=True)
    chlora_temp = chlora_temp.drop("index_right", axis=1)

    # append processed data to previously created dataframe
    chlora = chlora.append(chlora_temp[["district", "year", "month", "chlora"]]).reset_index(
        drop=True
    )

In [ ]:
%%time

# rename column because of 10 characters column name limit in shapefile format
chlora = chlora.rename(columns={"chlora": "chl"})

# aggregate data and add lag variables
chlora_district_level = process_ecv(chlora, "chl")

In [ ]:
# add unbuffered district geometries
chlora_district_level = pd.merge(
    chlora_district_level, outbreak_geometries, how="inner", on="district"
).reset_index(drop=True)

In [ ]:
# transform dataframe to geodataframe
chlora_district_level = gpd.GeoDataFrame(
    chlora_district_level, crs="EPSG:7755", geometry=chlora_district_level["geometry"]
)

Check out the processed file to make sure the output is as desired.

In [ ]:
chlora_district_level.shape

In [ ]:
chlora_district_level.info()

In [ ]:
chlora_district_level.head()

In [ ]:
chlora_district_level["district"].nunique()

In [ ]:
# only coastal districts that interact with the coverage of chlora are included
chlora_district_level[["geometry"]].drop_duplicates().to_crs(epsg=4326).plot()

In [ ]:
# chlora is included for coastal districts only, here shown exemplary for one month
chlora_temp[chlora_temp["month"] == 1].to_crs(epsg=4326).plot()

In [ ]:
# free memory
del [chlora_temp, chlora]
gc.collect()

### Land surface temperature

First, we load a sample of the data and explore it.

In [ ]:
path = "../data/land_surface_temperature"

In [ ]:
file = "monthly_lst_" + str(years[0]) + ".shp"

In [ ]:
%%time

lst = gpd.read_file(os.path.join(path, file))

In [ ]:
lst.shape

In [ ]:
lst.info()

In [ ]:
lst.head()

Now we process all the data.

In [ ]:
%%time

lst = pd.DataFrame(columns=["district", "year", "month", "lst"])

for year in years:
    print("Processing {}...".format(year))

    # read data into geodataframe
    lst_temp = gpd.read_file(os.path.join(path, "monthly_lst_" + str(year) + ".shp"))

    # reproject data to local crs
    lst_temp = lst_temp.to_crs(epsg=7755)

    # spatial join with unbuffered districts because lst is a terrestrial variable
    lst_temp = lst_temp.sjoin(
        outbreak_geometries_buffered, how="inner", predicate="within"
    ).reset_index(drop=True)
    lst_temp = lst_temp.drop("index_right", axis=1)

    # append processed data to previously created dataframe
    lst = lst.append(lst_temp[["district", "year", "month", "lst"]]).reset_index(drop=True)

In [ ]:
%%time

# aggregate data and add lag variables
lst_district_level = process_ecv(lst, "lst")

In [ ]:
# add unbuffered district geometries
lst_district_level = pd.merge(
    lst_district_level, outbreak_geometries, how="inner", on="district"
).reset_index(drop=True)

In [ ]:
# transform dataframe to geodataframe
lst_district_level = gpd.GeoDataFrame(
    lst_district_level, crs="EPSG:7755", geometry=lst_district_level["geometry"]
)

Check out the processed file to make sure the output is as desired.

In [ ]:
lst_district_level.shape

In [ ]:
lst_district_level.info()

In [ ]:
lst_district_level.head()

In [ ]:
lst_district_level["district"].nunique()

In [ ]:
# not only coastal districts that interact with the coverage of lst are included
lst_district_level[["geometry"]].drop_duplicates().to_crs(epsg=4326).plot()

In [ ]:
# lst is included not for coastal districts only, here shown exemplary for one month
lst_temp[lst_temp["month"] == 1].to_crs(epsg=4326).plot()

In [ ]:
# free memory
del [lst_temp, lst]
gc.collect()

## Merge cholera outbreaks and ECV data

Check distinct districts

In [ ]:
outbreaks["district"].nunique()

In [ ]:
sss_district_level["district"].nunique()

In [ ]:
chlora_district_level["district"].nunique()

In [ ]:
lst_district_level["district"].nunique()

Check shapes

In [ ]:
outbreaks.shape

In [ ]:
sss_district_level.shape

In [ ]:
chlora_district_level.shape

In [ ]:
lst_district_level.shape

In [ ]:
# merge cholera outbreaks and ecv data on district, year and month
train_test = pd.merge(
    outbreaks,
    sss_district_level.drop("geometry", axis=1),
    how="left",
    on=["district", "year", "month"],
)
train_test = pd.merge(
    train_test,
    chlora_district_level.drop("geometry", axis=1),
    how="left",
    on=["district", "year", "month"],
)
train_test = pd.merge(
    train_test,
    lst_district_level.drop("geometry", axis=1),
    how="left",
    on=["district", "year", "month"],
)
train_test.shape

In [ ]:
train_test["district"].nunique()

In [ ]:
train_test.info()

In [ ]:
train_test.head()

## Map district to coastal location

Next, we need to figure out which district is on the west and which is on the east coast of India. The following is a rough approach that should suffice for now. The southernmost point of mainland India is Cape Comorin. We check whether each district's centroid is east or west of Cape Comorin and thereby decide which coastal location to assign to each district.

In [ ]:
# geocode Cape Comorin
cape_comorin = geocode(
    "Cape Comorin", provider="nominatim", user_agent="cholera_risk_modeling", timeout=4
)

In [ ]:
cape_comorin.crs

The CRS of Cape Comoring and our data need to match.

In [ ]:
# reproject data to local crs
cape_comorin = cape_comorin.to_crs(epsg=7755)
train_test = train_test.to_crs(epsg=7755)

In [ ]:
cape_comorin.crs == train_test.crs

In [ ]:
cape_comorin

In [ ]:
# check the coordinates of Cape Comorin
split_point = cape_comorin.loc[0, "geometry"]
print(split_point)
print(split_point.x)
print(split_point.y)

In [ ]:
# get district centroids
train_test["centroid"] = train_test.centroid

In [ ]:
# check whether district centroids are west or east of Cape Comorin
train_test["location"] = train_test.apply(
    lambda x: "west" if x["centroid"].x < split_point.x else "east", axis=1
)

In [ ]:
train_test.head()

In [ ]:
# visually check whether the assignment makes intuitive sense
fig, ax = plt.subplots(figsize=(12, 8))
train_test[["geometry", "location"]].drop_duplicates().to_crs(epsg=4326).plot(
    ax=ax, column="location"
)
cape_comorin.to_crs(epsg=4326).plot(ax=ax, color="red")
plt.tight_layout()

In [ ]:
# drop centriod column because it is not needed anymore
train_test = train_test.drop("centroid", axis=1)

## Map month to season

Next, we need to map the months to their respective season according to the definition of the [India Meteorological Department](https://mausam.imd.gov.in/).

In [ ]:
# create dictionary with month to season mapping
seasons = {
    1: "winter",
    2: "winter",
    3: "pre_mnsn",
    4: "pre_mnsn",
    5: "pre_mnsn",
    6: "mnsn",
    7: "mnsn",
    8: "mnsn",
    9: "mnsn",
    10: "post_mnsn",
    11: "post_mnsn",
    12: "post_mnsn",
}

In [ ]:
# apply mapping
train_test["season"] = train_test["month"].map(seasons)

In [ ]:
train_test.head()

## One-hot encode categorical features

We also need to one-hot encode the categorical features in our data.

In [ ]:
location_dummies = pd.get_dummies(train_test["location"])

In [ ]:
train_test = pd.concat([train_test, location_dummies], axis=1).drop("location", axis=1)

In [ ]:
season_dummies = pd.get_dummies(train_test["season"])

In [ ]:
train_test = pd.concat([train_test, season_dummies], axis=1).drop("season", axis=1)

In [ ]:
train_test.head()

## Save final dataset

Finally, we save our data in shapefile format.

In [ ]:
%%time

train_test.to_file("../data/train_test.shp")